# 🚀 Model Export — Streamlit için
Bu notebook Kaggle'daki modellerini Streamlit uygulaması için dışa aktarır.

**Çalıştırma sırası:** Bu notebook'u 6 aşamalık notebook'un hemen ardından çalıştır.

---

In [ ]:
import pickle, os, re
import numpy as np
import pandas as pd
from scipy.sparse import load_npz, hstack, csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

KLASOR = '/kaggle/working/'
print('✅ Hazır.')

## Adım 1 — Veriyi yükle ve temizle

In [ ]:
HAM_XLSX = '/kaggle/input/datasets/eminenuraktas1/turkish-math-comments-labeled/matematik_yorumlari_final_etiketli.xlsx'

# Temiz CSV varsa yükle, yoksa ham veriyi temizle
TEMIZ_CSV = KLASOR + 'matematik_ozellikli.csv'
if os.path.exists(TEMIZ_CSV):
    df = pd.read_csv(TEMIZ_CSV, encoding='utf-8-sig')
    print(f'✅ Temiz CSV yüklendi: {df.shape}')
else:
    df = pd.read_excel(HAM_XLSX)
    DUYGU_MAP = {'OLumsuz': 'Olumsuz', 'Olumlu ': 'Olumlu', ' Olumlu': 'Olumlu'}
    df['duygu'] = df['duygu'].str.strip().replace(DUYGU_MAP)
    df = df.dropna(subset=['yorum'])
    df['yorum'] = df['yorum'].astype(str).str.strip()
    df = df[df['yorum'] != ''].drop_duplicates(subset=['yorum']).reset_index(drop=True)

    STOP = {'ve','veya','ama','ile','bir','bu','şu','o','da','de','mi','mı',
            'için','gibi','kadar','daha','en','çok','var','yok','ben','sen'}

    def temizle(m):
        m = re.sub(r'http\S+|www\.\S+|@\w+', '', str(m))
        m = re.sub(r'[^\w\sşğıöüçŞĞIÖÜÇ.,!?]', ' ', m, flags=re.UNICODE)
        m = re.sub(r'(.)\1{2,}', r'\1\1', m)
        return re.sub(r'\s+', ' ', m).strip()

    def nlp(m):
        m = temizle(m).lower()
        m = re.sub(r'[^\w\sşğıöüç]', ' ', m, flags=re.UNICODE)
        return ' '.join([k for k in m.split() if k not in STOP and len(k) > 2])

    df['yorum_temiz'] = df['yorum'].apply(temizle)
    df['yorum_nlp']   = df['yorum_temiz'].apply(nlp)
    df['duygu_binary']         = (df['duygu'] != 'Nötr').astype(int)
    df['sinav_kaygisi_binary'] = (df['sinav_kaygisi'].str.lower() == 'var').astype(int)
    df['yorum_tipi_binary']    = (df['yorum_tipi'] != 'Genel').astype(int)
    print(f'✅ Ham veri temizlendi: {df.shape}')

print('Sütunlar:', df.columns.tolist())

## Adım 2 — TF-IDF eğit

In [ ]:
# TF-IDF varsa yükle, yoksa eğit
TFIDF_PKL = KLASOR + 'tfidf_ngram.pkl'
if os.path.exists(TFIDF_PKL):
    with open(TFIDF_PKL, 'rb') as f:
        tfidf_ngram = pickle.load(f)
    print('✅ Mevcut TF-IDF yüklendi.')
else:
    tfidf_ngram = TfidfVectorizer(
        ngram_range=(1, 2), max_features=10000,
        min_df=2, max_df=0.90, sublinear_tf=True
    )
    tfidf_ngram.fit(df['yorum_nlp'].fillna(''))
    print('✅ TF-IDF eğitildi.')

X_tfidf = tfidf_ngram.transform(df['yorum_nlp'].fillna(''))
print(f'TF-IDF matris: {X_tfidf.shape}')

## Adım 3 — Sınav Kaygısı Modeli Eğit & Kaydet

In [ ]:
# Hedef
y_kaygi = df['sinav_kaygisi_binary'].values

# Train/test split
idx = np.arange(len(df))
idx_train, idx_test = train_test_split(idx, test_size=0.15, stratify=y_kaygi, random_state=42)

X_train = X_tfidf[idx_train]
X_test  = X_tfidf[idx_test]
y_train = y_kaygi[idx_train]
y_test  = y_kaygi[idx_test]

# En iyi model varsa yükle
BEST_PKL = KLASOR + 'best_model.pkl'
if os.path.exists(BEST_PKL):
    with open(BEST_PKL, 'rb') as f:
        kaygi_model = pickle.load(f)
    print('✅ Mevcut best_model.pkl yüklendi.')
else:
    print('Model bulunamadı, GridSearch ile eğitiliyor...')
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    grid = GridSearchCV(
        LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
        {'C': [0.1, 1.0, 5.0, 10.0], 'class_weight': ['balanced', None]},
        cv=cv, scoring='roc_auc', n_jobs=-1
    )
    grid.fit(X_train, y_train)
    kaygi_model = grid.best_estimator_
    print(f'✅ Eğitildi. En iyi params: {grid.best_params_}')

# Test skoru
from sklearn.metrics import roc_auc_score, f1_score
try:
    y_pred = kaygi_model.predict(X_test)
    y_prob = kaygi_model.predict_proba(X_test)[:, 1]
    print(f'Test ROC-AUC : {roc_auc_score(y_test, y_prob):.4f}')
    print(f'Test F1      : {f1_score(y_test, y_pred, average="weighted"):.4f}')
except Exception as e:
    # Model boyut uyumsuzluğu (meta+kat özellikler eklendiyse)
    print(f'Not: Model farklı boyutta eğitilmiş ({e})')
    print('Yeniden eğitiliyor...')
    kaygi_model = LogisticRegression(C=10.0, class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1)
    kaygi_model.fit(X_train, y_train)
    y_pred = kaygi_model.predict(X_test)
    y_prob = kaygi_model.predict_proba(X_test)[:, 1]
    print(f'Test ROC-AUC : {roc_auc_score(y_test, y_prob):.4f}')

## Adım 4 — Duygu Modeli Eğit & Kaydet

In [ ]:
# Duygu modeli — çok sınıflı LR
df['duygu_temiz'] = df['duygu'].str.strip().str.capitalize()
y_duygu = df['duygu_temiz'].values

duygu_model = LogisticRegression(
    C=5.0, class_weight='balanced',
    max_iter=1000, random_state=42,
    multi_class='multinomial', solver='lbfgs', n_jobs=-1
)
duygu_model.fit(X_tfidf, y_duygu)

from sklearn.metrics import classification_report
y_duygu_pred = duygu_model.predict(X_tfidf[idx_test])
print('Duygu modeli eğitildi.')
print(classification_report(y_duygu[idx_test], y_duygu_pred))

## Adım 5 — Dosyaları Kaydet (Streamlit için)

In [ ]:
import zipfile

# 1. Kaygı modeli
with open(KLASOR + 'best_model.pkl', 'wb') as f:
    pickle.dump(kaygi_model, f)
print('✅ best_model.pkl kaydedildi')

# 2. Duygu modeli
with open(KLASOR + 'duygu_model.pkl', 'wb') as f:
    pickle.dump(duygu_model, f)
print('✅ duygu_model.pkl kaydedildi')

# 3. TF-IDF
with open(KLASOR + 'tfidf_ngram.pkl', 'wb') as f:
    pickle.dump(tfidf_ngram, f)
print('✅ tfidf_ngram.pkl kaydedildi')

# 4. Hepsini tek ZIP'e sıkıştır (indirmesi kolay)
ZIP_YOLU = KLASOR + 'streamlit_modeller.zip'
with zipfile.ZipFile(ZIP_YOLU, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(KLASOR + 'best_model.pkl',   'model/best_model.pkl')
    zf.write(KLASOR + 'duygu_model.pkl',  'model/duygu_model.pkl')
    zf.write(KLASOR + 'tfidf_ngram.pkl',  'model/tfidf_ngram.pkl')

boyut = os.path.getsize(ZIP_YOLU) / 1024 / 1024
print(f'\n📦 streamlit_modeller.zip oluşturuldu: {boyut:.1f} MB')
print('\n✅ Output sekmesinden bu ZIP dosyasını indir!')
print('   Sonra matematik_app/model/ klasörüne çıkar.')

In [ ]:
# Doğrulama testi — model gerçekten çalışıyor mu?
test_yorumlar = [
    'Hocam ayt sınavına çok az kaldı, geometriden çok korkuyorum yardım edin',
    'Teşekkürler hocam, türevi sonunda anladım, çok güzel anlatıyorsunuz',
    'Bu sorunun 3. adımını anlayamadım, farklı yöntemle çözebilir misiniz',
]

def test_temizle(m):
    STOP = {'ve','veya','ama','ile','bir','bu','şu','o','da','de','mi','mı','için','gibi'}
    m = re.sub(r'http\S+|@\w+|[^\w\sşğıöüçŞĞIÖÜÇ.,!?]', ' ', str(m), flags=re.UNICODE)
    m = m.lower()
    return ' '.join([k for k in m.split() if k not in STOP and len(k) > 2])

print('── Model Doğrulama Testi ──────────────────────────────')
for yorum in test_yorumlar:
    nlp_yorum = test_temizle(yorum)
    x = tfidf_ngram.transform([nlp_yorum])
    kaygi_pred = kaygi_model.predict(x)[0]
    kaygi_prob = kaygi_model.predict_proba(x)[0][1]
    duygu_pred = duygu_model.predict(x)[0]
    print(f'  Yorum  : {yorum[:60]}...')
    print(f'  Kaygı  : {"VAR" if kaygi_pred else "YOK"} (%{kaygi_prob*100:.1f})')
    print(f'  Duygu  : {duygu_pred}')
    print()